# UNBLIND_02 — Sub-DLA CDDF ([19.5, 20.3), the LLS -> DLA band)

## 🔴 READ ME FIRST

* This notebook plots the **UNBLINDED real-LOA sub-DLA measurement** (DESI DR2 Loa
  main-dark). Real-LOA result values (dN/dX, Omega, f(N), per-z tables) are **PRIVATE**.
* **CLEAR ALL OUTPUTS before committing** to the code repo. The committed `.ipynb`
  must contain **zero executed outputs** — rendered figures encode real-LOA values in
  their axis ticks.
* **Figures belong in the private notes repo** (`jibanMat/desi_gpy_dla_notes`), never
  in the code repo. This notebook never writes a figure into the repo.
* Every plotted quantity is **read from the loaded artifact at render time** — no
  real-LOA number is hard-coded. Mock (2LPT-0) values are public and may appear where
  they are explicitly labelled *mock*.
* The real-LOA sub-DLA artifact **does not exist yet**: `run_subdla_headline_full.py
  --real-loa` (Team2-A's routine) must be run first, **under PI approval**. Until then,
  set `SMOKE_TEST=True` to exercise the plotting code against the committed 2LPT-0 mock.


## Artifact schema + parameters

Two schemas are handled:

* **Headline (real-LOA)** — top level `['measurement','metadata','perz_fN','zbins']`;
  `measurement[<limit>]['dndx'|'omega']` each with `integrated`
  (`MAP,q025,q16,q84,q975,std`) and `perz` (list per z-bin); `perz_fN` with
  `logN_centers, zbins, z_extrapolated, z_thin, truth_counts_perz, perz`.
* **Mock validation** (`CDDF_analysis/hbi/subdla_mock_validation.json`) — top level
  `['metadata','per_bin','integrated']`; `per_bin[fp]` per-0.1-dex list
  (`blo,bhi,f_est,f_tru,dndx_est,dndx_tru,r0`); `integrated[fp]` with
  `dndx_est_195_203, omega_est_195_203, r0_*` etc. This is a **z-marginal** public
  mock — it has **no per-z / zbins / MC band**, so per-z + z>4 cells degrade
  gracefully to a labelled note when the mock is loaded.

Design decisions honoured here:
* `[19.5,19.7)` is **formally non-identifiable** on a 19.5-floored catalog — those bins
  are hatched, **never plotted as a measurement**.
* Extrapolation is flagged with **three distinct masks**:
  `beyond_calibration` (bin lower edge above the mock truth cap `max_truth_z` -> no
  completeness support), `beyond_v2_fit` (bin entirely above the mean-flux fit ceiling
  `v2_z_fit_hi=3.5`), `partial_truth_support` (bin straddles `max_truth_z`).


In [ ]:
import os, sys, json, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# PARAMETERS — the only knobs. Every plotted quantity is READ from the artifact.
# ---------------------------------------------------------------------------
# Committed default = real-LOA intent. Set SUBDLA_SMOKE_TEST=1 in the environment
# (or flip this to True by hand) to exercise ALL plotting code on the committed
# 2LPT-0 MOCK (public values, NOT real-LOA).
SMOKE_TEST = os.environ.get("SUBDLA_SMOKE_TEST", "0") == "1"


def _find_repo_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, ".git")):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.getcwd())


REPO_ROOT = os.environ.get("SUBDLA_REPO_ROOT", _find_repo_root())

# Real-LOA sub-DLA headline JSON — produced by run_subdla_headline_full.py --real-loa
# (PI-GATED). Real-LOA values live on SCRATCH, never the repo. Set to that --out path.
REAL_LOA_ARTIFACT = os.environ.get(
    "SUBDLA_REAL_LOA_ARTIFACT",
    "/scratch/cavestru_root/cavestru0/mfho/cddf_o3_realdata/subdla/subdla_headline_loa0.json")
# OPTIONAL purity_mixture real-LOA headline (the reported FP-bracket partner).
REAL_LOA_ARTIFACT_PM = os.environ.get(
    "SUBDLA_REAL_LOA_ARTIFACT_PM",
    "/scratch/cavestru_root/cavestru0/mfho/cddf_o3_realdata/subdla/subdla_headline_pm.json")
# OPTIONAL DLA-tier headline (UNBLIND_01 / Team 1) for the continuity panel.
DLA_ARTIFACT = os.environ.get("SUBDLA_DLA_ARTIFACT", "")

# committed MOCK artifact (public 2LPT-0 recovery ratios) — always available.
MOCK_ARTIFACT = os.path.join(REPO_ROOT, "CDDF_analysis", "hbi", "subdla_mock_validation.json")
# edge-migration systematic (stamped by a sibling workstream; may be absent).
EDGE_SYSTEMATIC_JSON = os.environ.get(
    "SUBDLA_EDGE_SYSTEMATIC_JSON",
    os.path.join(REPO_ROOT, "CDDF_analysis", "hbi", "subdla_edge_systematic.json"))

# where the smoke run may drop PNGs FOR INSPECTION (never the repo). None => no save.
FIG_OUT_DIR = os.environ.get("SUBDLA_FIG_OUT_DIR") or None

# [19.5,19.7): bins with upper edge <= this are NON-IDENTIFIABLE on a 19.5-floored
# catalog (Track A). Honoured unless the artifact emits an explicit mask.
NONIDENT_LOGN_HI = 19.7
# z-bin extrapolation thresholds (fallbacks; the artifact's metadata overrides).
DEFAULT_V2_Z_FIT_HI = 3.5

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"SMOKE_TEST     = {SMOKE_TEST}")
print(f"MOCK_ARTIFACT  = {MOCK_ARTIFACT}")


In [ ]:
import os, sys
_REPO = os.path.abspath(os.environ.get("UNBLIND_REPO_ROOT", os.getcwd()))
while _REPO != "/" and not os.path.isdir(os.path.join(_REPO, ".git")):
    _REPO = os.path.dirname(_REPO)
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)

# ---------------------------------------------------------------------------
# PROVENANCE GUARD — must FAIL CLOSED.
#
# There was a `# TEMPORARY STUB` fallback here. It was removed on 2026-07-09 after an
# adversarial review showed it fails OPEN: the stub did no git inspection at all (a bare
# `^[0-9a-f]{7,40}$` regex), so it accepted a fabricated SHA and the ORPHANED london-0
# stamp `cff73cb` as RE_DERIVABLE. The only signal was a print, which disappears the moment
# outputs are cleared for commit. An unblinding notebook's central safety mechanism must
# raise when the real guard cannot be imported -- never substitute a weaker one.
# ---------------------------------------------------------------------------
from CDDF_analysis.unblind import check_artifact, load_headline, carried_systematics  # noqa: E402


In [ ]:
# ---- LOAD (guard runs inside load_and_adapt, BEFORE any data is touched) ----
if SMOKE_TEST:
    print("=" * 78)
    print("SMOKE-TEST MODE — plotting the committed 2LPT-0 MOCK (PUBLIC values, NOT real-LOA).")
    print("  artifact:", MOCK_ARTIFACT)
    print("=" * 78)
    ART = load_and_adapt(MOCK_ARTIFACT)
else:
    if not os.path.exists(REAL_LOA_ARTIFACT):
        msg = ("\n" + "=" * 78 + "\n"
               "REAL-LOA SUB-DLA ARTIFACT NOT FOUND:\n"
               f"    {REAL_LOA_ARTIFACT}\n\n"
               "This notebook plots the UNBLINDED real-LOA sub-DLA measurement, which does\n"
               "NOT exist yet. Under PI approval, first run Team2-A's headline routine:\n\n"
               "    python run_subdla_headline_full.py --real-loa --out <path>\n\n"
               "then set REAL_LOA_ARTIFACT (or $SUBDLA_REAL_LOA_ARTIFACT) to that --out path.\n"
               "To exercise the plotting code on the committed MOCK instead, set\n"
               "SMOKE_TEST=True (or $SUBDLA_SMOKE_TEST=1).\n" + "=" * 78)
        print(msg)
        raise FileNotFoundError(f"real-LOA sub-DLA artifact absent: {REAL_LOA_ARTIFACT}")
    print(f"[load] real-LOA artifact: {REAL_LOA_ARTIFACT}")
    ART = load_and_adapt(REAL_LOA_ARTIFACT)

print(f"[load] schema={ART['schema']}  is_mock={ART['is_mock']}  "
      f"headline_fp={ART['headline_fp']}  fp_models={ART['fp_models']}")


## Figure 1 — Differential f(N) over [19.7, 20.3)

**Shows:** the z-marginal sub-DLA column-density distribution f(N) with the recovered
estimate over the **identifiable** band [19.7, 20.3). The `[19.5, 19.7)` columns are
**hatched out** — on a 19.5-floored catalog that mass is formally non-identifiable and
is **never** shown as a measurement. On the mock, the 2LPT-0 truth curve is overlaid.

**Falsified if:** the recovered f(N) sits outside the 68/95 band of the truth in the
identifiable band (mock), or the hatched columns are ever treated as a measured point.


In [ ]:
def figure_1_differential_fN(A):
    fig, ax = plt.subplots(figsize=(7.4, 5.0))
    hi = NONIDENT_LOGN_HI
    if A["is_mock"]:
        fp = A["headline_fp"]
        d = A["diff_marginal"][fp]
        mid, f, ft = d["logN_mid"], d["f"], d["f_truth"]
        mask_ni = nonident_mask_edges(d["logN_lo"], d["logN_hi"])
        ok = ~mask_ni
        ax.step(mid[ok], f[ok], where="mid", color="#1f77b4", lw=2, label=f"{fp} f(N) est (mock)")
        ax.plot(mid[ok], f[ok], "o", color="#1f77b4", ms=6)
        ax.step(mid[ok], ft[ok], where="mid", color="k", lw=1.2, ls="--", label="2LPT-0 truth f(N)")
        lo_ni = float(d["logN_lo"][mask_ni].min())
        ax.axvspan(lo_ni, hi, facecolor="none", edgecolor="crimson", hatch="xx", alpha=0.6, zorder=0)
        ymid = float(np.sqrt(np.nanmin(f[ok]) * np.nanmax(f[ok])))
        ax.text(0.5 * (lo_ni + hi), ymid, "NON-IDENTIFIABLE\n[19.5,19.7)\n(19.5-floored catalog)",
                color="crimson", ha="center", va="center", fontsize=8)
        print("[fig1] MOCK point estimate (no MC band in the validation schema).")
    else:
        pf = A["perz_fN"]
        centers = np.asarray(pf["logN_centers"], float)
        mask_ni = nonident_mask_centers(centers)
        perz = pf.get("perz", [])
        empties = perz_fN_empty_flags(pf)
        nz = max(len(perz), 1)
        plotted, skipped_empty = [], []
        for i, zc in enumerate(perz):
            if zc.get("extrapolated"):     # no calibration support -> NOT a measurement
                continue
            f = np.asarray(zc.get("f", []), float)
            if (i < len(empties) and empties[i]) or np.isfinite(f).sum() == 0:
                skipped_empty.append(i)    # per-z CDDF absent above v2_z_fit_hi
                continue
            col = plt.cm.viridis(i / max(nz - 1, 1))
            lo68, hi68 = np.asarray(zc["f68_lo"], float), np.asarray(zc["f68_hi"], float)
            sel = (~mask_ni) & np.isfinite(f) & (f > 0)
            ax.plot(centers[sel], f[sel], "-", color=col, lw=1.6, label=f"z~{zc.get('z'):.2f}")
            ax.fill_between(centers[sel], lo68[sel], hi68[sel], color=col, alpha=0.20)
            plotted.append(i)
        lo_ni = float(centers[mask_ni].min()) if mask_ni.any() else 19.5
        ax.axvspan(lo_ni, hi, facecolor="none", edgecolor="crimson", hatch="xx", alpha=0.6, zorder=0)
        print(f"[fig1] REAL-LOA per-z f(N|z) MAP + 68% band; plotted z_idx={plotted}; "
              f"skipped extrapolated + EMPTY-per-z-CDDF z_idx={skipped_empty} (v2_z_fit_hi).")
    ax.axvline(hi, color="crimson", ls=":", lw=1)
    ax.set_yscale("log")
    ax.set_xlabel(r"$\log_{10} N_{\rm HI}$")
    ax.set_ylabel(r"$f(N_{\rm HI})$")
    ax.set_xlim(19.45, 20.35)
    ax.set_title("Sub-DLA differential f(N): measurement over [19.7,20.3)\n"
                 "[19.5,19.7) hatched = NON-IDENTIFIABLE (never a measurement)")
    ax.legend(fontsize=8, loc="best")
    ax.grid(alpha=0.25, which="both")
    _finish(fig, "fig1_subdla_differential_fN.png")


figure_1_differential_fN(ART)


## Figure 1b — Per-z sub-DLA CDDF panels (+ extrapolation masks)

**Shows:** one f(N|z) panel per z-bin over [19.7, 20.3) with 68/95 bands, the
`[19.5,19.7)` columns hatched-out, and each panel decorated by its **extrapolation
masks**: `beyond_calibration` (red, no completeness support beyond the mock truth cap),
`partial_truth_support` (amber, bin straddles the cap), `beyond_v2_fit` (gold, entirely
above the mean-flux fit ceiling z>3.5). Panel border colour + background tint encode the
dominant mask; all applicable masks are listed in-panel.

Headline schema only — the public mock is z-marginal and renders a labelled placeholder.

**`perz_fN_empty` (a 4th, derived flag):** the per-z f(N|z) reduction has **no finite
support above `v2_z_fit_hi=3.5`** — `perz_fN.perz[i].f` is entirely NaN for z=3.75 and
z=4.125 — even though `measurement[<lim>].perz` still carries genuine dN/dX & Omega there.
Such panels are drawn **grey with an explicit "NO PER-z CDDF ABOVE z=3.5" annotation** —
**never blank** (a blank axis and a null measurement look identical to a reader). **Trap:**
the z=3.75 bin has `extrapolated=False` **and** an empty f(N).

**Falsified if:** a masked/empty bin is drawn identically to a calibrated/supported bin, an
empty-f panel is rendered blank, or the z=3.75 bin is shown as fully supported.


In [ ]:
def figure_1b_perz_panels(A):
    if A["is_mock"] or A.get("perz_fN") is None:
        print("[fig1b] Per-z panels require the HEADLINE (real-LOA) schema (perz_fN + zbins).")
        print("        MOCK validation schema is z-marginal (no per-z CDDF) — placeholder shown.")
        _placeholder("Per-z sub-DLA CDDF panels\n(headline schema only; the mock has no per-z data)")
        return
    pf = A["perz_fN"]
    centers = np.asarray(pf["logN_centers"], float)
    zbins = pf.get("zbins") or A.get("zbins") or A["metadata"].get("zbins")
    v2hi = float(A["metadata"].get("v2_z_fit_hi", DEFAULT_V2_Z_FIT_HI))
    masks = compute_z_masks(A["metadata"], zbins)
    empties = perz_fN_empty_flags(pf)          # 4th DERIVED per-z flag
    mask_ni = nonident_mask_centers(centers)
    perz = pf.get("perz", [])
    nz = len(perz)
    ncol = min(nz, 3) if nz else 1
    nrow = int(np.ceil(nz / ncol)) if nz else 1
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.3 * ncol, 3.7 * nrow), squeeze=False)
    populated, empty = [], []
    for k, zc in enumerate(perz):
        ax = axes[k // ncol][k % ncol]
        m = masks[k] if k < len(masks) else dict()
        m["perz_fN_empty"] = bool(empties[k]) if k < len(empties) else False
        st = z_mask_style(m)
        for sp in ax.spines.values():
            sp.set_edgecolor(st["edge"])
            sp.set_linewidth(2.2)
        ax.set_title(f"z~{zc.get('z'):.2f}  [{m.get('z_lo', float('nan')):.2f},"
                     f"{m.get('z_hi', float('nan')):.2f})", fontsize=9)
        tags = mask_tags(m)
        f = np.asarray(zc.get("f", []), float)
        if m["perz_fN_empty"] or np.isfinite(f).sum() == 0:
            # NEVER blank: a blank axis and a null measurement are indistinguishable.
            empty.append(k)
            ax.set_facecolor("#e6e6e6")
            ax.text(0.5, 0.56,
                    f"NO PER-z CDDF ABOVE z={v2hi:.1f}\n(fine z-fit grid ends at\n"
                    f"v2_z_fit_hi={v2hi:.1f})",
                    transform=ax.transAxes, ha="center", va="center", fontsize=8.5,
                    color="dimgray", weight="bold")
            ax.text(0.5, 0.22, "dN/dX & Omega DO exist here\n(measurement[<lim>].perz)",
                    transform=ax.transAxes, ha="center", va="center", fontsize=7, color="#333")
            ax.axvspan(19.5, NONIDENT_LOGN_HI, facecolor="none", edgecolor="crimson",
                       hatch="xx", alpha=0.35, zorder=0)
            ax.text(0.5, 0.02, st["banner"] + "\nmasks: " + ", ".join(tags),
                    transform=ax.transAxes, ha="center", va="bottom", fontsize=6.3,
                    color=st["edge"])
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        populated.append(k)
        if st["tint"] != "none":
            ax.set_facecolor(st["tint"])
        lo68, hi68 = np.asarray(zc["f68_lo"], float), np.asarray(zc["f68_hi"], float)
        lo95, hi95 = np.asarray(zc["f95_lo"], float), np.asarray(zc["f95_hi"], float)
        sel = (~mask_ni) & np.isfinite(f) & (f > 0)
        ax.fill_between(centers[sel], lo95[sel], hi95[sel], color=st["edge"], alpha=0.12)
        ax.fill_between(centers[sel], lo68[sel], hi68[sel], color=st["edge"], alpha=0.28)
        ax.plot(centers[sel], f[sel], ls=st["ls"], color=st["edge"], lw=1.8,
                marker=st["marker"], mfc=st["mfc"], ms=4)
        ax.axvspan(19.5, NONIDENT_LOGN_HI, facecolor="none", edgecolor="crimson",
                   hatch="xx", alpha=0.6, zorder=0)
        ax.text(0.5, 0.02, st["banner"] + (("\nmasks: " + ", ".join(tags)) if tags else ""),
                transform=ax.transAxes, ha="center", va="bottom", fontsize=6.3, color=st["edge"])
        ax.set_yscale("log")
        ax.set_xlim(19.45, 20.35)
        ax.grid(alpha=0.2, which="both")
    for j in range(nz, nrow * ncol):
        axes[j // ncol][j % ncol].axis("off")
    fig.suptitle("Per-z sub-DLA CDDF f(N|z) over [19.7,20.3) — [19.5,19.7) hatched non-identifiable;\n"
                 "border/tint = extrapolation mask; grey 'NO PER-z CDDF' = perz_fN_empty (v2_z_fit_hi)")
    _finish(fig, "fig1b_perz_panels.png")
    print(f"[fig1b] populated per-z CDDF panels (z_idx): {populated}")
    print(f"[fig1b] EMPTY per-z CDDF panels (z_idx, no f above v2_z_fit_hi={v2hi:.1f}): {empty}")
    if empty:
        print("[fig1b] TRAP: an empty per-z CDDF bin can still have extrapolated=False "
              "(e.g. z~3.75) yet carry genuine dN/dX & Omega in measurement[<lim>].perz.")


figure_1b_perz_panels(ART)


## z > 4 — unvalidated extrapolation (READ CAREFULLY)

The 2LPT-0 mock truth **caps at z ~ 3.5–3.79** (`max_truth_z`). Therefore the **z > 4
sub-DLA bin is validated by NO mock** — its completeness `g(N,z)` is a pure, **unbounded**
extrapolation, and the statistical MC band does **not** cover this systematic.

**Compounding — three distinct defects at once** in the z > 4 sub-DLA band:
1. **in logN:** the `[19.5,19.7)` mass is **non-identifiable** (19.5-floored catalog);
2. **in z:** completeness `g(N,z)` is a **pure, unbounded extrapolation** (no mock truth);
3. **in data:** the per-z CDDF `f(N|z)` is **absent entirely** (`v2_z_fit_hi=3.5` fine grid,
   `perz_fN_empty`).

Only integrated **dN/dX** and **Omega** exist at z > 4 (`measurement[<lim>].perz`); there is
**no** per-z `f(N|z)` curve to plot, and none of the three defects is inside the MC band.
Raising `v2_z_fit_hi` to recover the per-z CDDF is a **PI-gated science decision** — this
notebook never re-runs a headline to work around it.


In [ ]:
def zgt4_section(A):
    md = A["metadata"]
    print("=" * 78)
    print("z > 4 SUB-DLA EXTRAPOLATION")
    print("=" * 78)
    print(f"  mock truth cap   max_truth_z = {md.get('max_truth_z')}  (2LPT-0 caps at z~3.5-3.79)")
    print(f"  v2 fit ceiling   v2_z_fit_hi = {md.get('v2_z_fit_hi', DEFAULT_V2_Z_FIT_HI)}")
    pf = A.get("perz_fN")
    zbins = (pf or {}).get("zbins") or A.get("zbins") or md.get("zbins")
    empties = perz_fN_empty_flags(pf) if pf else []
    if zbins is None:
        print("  [note] MOCK schema has no zbins/perz_fN — cannot enumerate the z>4 bin here.")
    else:
        masks = compute_z_masks(md, zbins)
        zg = [(k, m) for k, m in enumerate(masks) if m["z_lo"] >= 4.0 - 1e-9]
        if not zg:
            print("  [note] no z-bin with lower edge >= 4.0 in this artifact's zbins.")
        for k, m in zg:
            pe = bool(empties[k]) if k < len(empties) else "n/a"
            print(f"  z-bin [{m['z_lo']:.2f},{m['z_hi']:.2f}): "
                  f"beyond_calibration={m['beyond_calibration']}  "
                  f"beyond_v2_fit={m['beyond_v2_fit']}  "
                  f"partial_truth_support={m['partial_truth_support']}  "
                  f"perz_fN_empty={pe}")
    print("  -> THREE DISTINCT DEFECTS compound in the z>4 sub-DLA band:")
    print("     (1) in logN: the [19.5,19.7) mass is NON-IDENTIFIABLE (19.5-floored catalog);")
    print("     (2) in z:    completeness g(N,z) is a PURE UNBOUNDED extrapolation (no truth);")
    print("     (3) in data: the per-z CDDF f(N|z) is ABSENT ENTIRELY (v2_z_fit_hi=3.5 fine grid).")
    print("  -> Only integrated dN/dX & Omega (measurement[<lim>].perz) exist at z>4; there is")
    print("     NO per-z f(N|z) curve to plot, and NONE of these three defects is inside the MC band.")


zgt4_section(ART)


## Figure 2 — Integrated dN/dX and Omega over [19.5, 20.3]

**Shows:** the integrated sub-DLA dN/dX and Omega with the **statistical MC band**
(68/95) and, drawn as a **separate, visually distinct** orange band, the
**edge-migration systematic** read from `subdla_edge_systematic.json` (a mock-derived
bracket over {floor-19.5, basis-pad 19.2/19.0, floor-19.0}; the final number is
**panel-adjudicated**). If that artifact is absent, only the MC band is drawn and a
prominent **TOTAL-ERROR-INCOMPLETE** warning is printed. On the mock, the point estimate
and 2LPT-0 truth are shown with the mock recovery ratio annotated.

**Falsified if:** the edge-migration band is folded into (drawn on top of) the MC band,
or the total error is presented as complete while `subdla_edge_systematic.json` is absent.


In [ ]:
def _load_edge_systematic():
    if not os.path.exists(EDGE_SYSTEMATIC_JSON):
        return None
    try:
        provenance_guard(EDGE_SYSTEMATIC_JSON)   # BEFORE touching data
        return _load_json(EDGE_SYSTEMATIC_JSON)
    except Exception as e:
        print(f"[fig2][WARN] edge-systematic JSON present but rejected by guard: {e}")
        return None


def _edge_frac(edge, kind):
    # fractional HALF-spread of the bracket R0 points relative to the headline R0.
    # NOT a prescribed number: the routine's own note defers the final value to the panel.
    try:
        res = edge["result"]
        cfgs = ("headline_floor19_5", "basis_pad_19_2", "basis_pad_19_0", "floor19_0_reference")
        pts = []
        for c in cfgs:
            blk = res.get(c)
            if not blk:
                continue
            v = blk.get(kind, {}).get("r0_band_195_203", {}).get("point")
            if v is not None and np.isfinite(v):
                pts.append(float(v))
        if len(pts) < 2:
            return None
        head = res["headline_floor19_5"][kind]["r0_band_195_203"]["point"]
        return float((max(pts) - min(pts)) / (2.0 * abs(head)))
    except Exception:
        return None


def figure_2_integrated(A):
    fp = A["headline_fp"]
    ig = A["integ"].get(fp, {})
    edge = _load_edge_systematic()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
    for ax, kind, scale, ylab in ((axes[0], "dndx", 1.0, r"$dN/dX\ [19.5,20.3)$"),
                                   (axes[1], "omega", 1e3, r"$10^3\,\Omega_{\rm HI}\ [19.5,20.3)$")):
        m = ig.get(kind, {})
        pt = m.get("point")
        if pt is None:
            ax.text(0.5, 0.5, "no integrated value in artifact", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_title(kind)
            continue
        x = 0.0
        if m.get("q16") is not None:
            ax.vlines(x, m["q025"] * scale, m["q975"] * scale, color="#1f77b4", lw=2, label="95% MC")
            ax.vlines(x, m["q16"] * scale, m["q84"] * scale, color="#1f77b4", lw=8, alpha=0.55,
                      label="68% MC")
        ax.plot(x, pt * scale, "o", color="#1f77b4", ms=9, mec="k", zorder=5,
                label=("mock est" if A["is_mock"] else "MAP"))
        if m.get("truth") is not None:
            ax.plot(x, m["truth"] * scale, "*", color="k", ms=16, zorder=6, label="2LPT-0 truth")
            if m.get("r0") is not None:
                ax.annotate(f"mock R0 = {m['r0']:.3f}", (x, pt * scale),
                            textcoords="offset points", xytext=(14, 0), fontsize=9)
        if edge is not None:
            frac = _edge_frac(edge, kind)
            if frac is not None:
                xe = 0.4
                ax.vlines(xe, pt * scale * (1 - frac), pt * scale * (1 + frac),
                          color="darkorange", lw=8, alpha=0.5,
                          label=f"edge-migration bracket (+/-{100*frac:.0f}% mock)")
                ax.plot(xe, pt * scale, "s", color="darkorange", ms=7, mec="k")
        else:
            print(f"[fig2][WARN] {kind}: subdla_edge_systematic.json ABSENT -> "
                  f"TOTAL ERROR INCOMPLETE (MC band only; edge-migration systematic missing).")
        ax.set_xlim(-0.5, 1.0)
        ax.set_xticks([])
        ax.set_ylabel(ylab)
        ax.grid(alpha=0.25)
        ax.set_title(kind)
        ax.legend(fontsize=8, loc="best")
    fig.suptitle("Integrated sub-DLA [19.5,20.3): statistical MC band vs (separate) edge-migration"
                 " systematic\n" + ("MOCK (point + truth; no MC band)" if A["is_mock"] else "REAL-LOA"))
    _finish(fig, "fig2_subdla_integrated.png")


figure_2_integrated(ART)


## Figure 3 — FP-model bracket (loa0 vs purity_mixture)

**Shows:** the **mock 2LPT-0 recovery ratios** R0 = est/truth for the integrated
[19.5,20.3) dN/dX and Omega, `loa0` vs `purity_mixture`, read from the committed mock
artifact. loa0 recovers the sub-DLA band far better than purity_mixture (which
over-subtracts sub-DLA -> DLA migration as a false positive) — this is **why loa0 is the
headline**. On real data, if a purity_mixture headline artifact is also provided, the
right panel overlays the two real integrated estimates side by side (no values echoed).

**Falsified if:** loa0's mock recovery R0 is not closer to 1 than purity_mixture's in
this band, undermining the headline FP choice.


In [ ]:
def figure_3_fp_bracket(A):
    mockA = A if A["is_mock"] else load_and_adapt(MOCK_ARTIFACT)  # committed, always available
    rec = mockA["fp_recovery"]
    fps = [fp for fp in ("loa0", "purity_mixture") if fp in rec]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    ax = axes[0]
    x = np.arange(len(fps))
    w = 0.35
    for xi, fp in zip(x, fps):
        ax.bar(xi - w / 2, rec[fp]["dndx_r0"], w, color="#1f77b4",
               label="dN/dX R0" if xi == 0 else None)
        ax.bar(xi + w / 2, rec[fp]["omega_r0"], w, color="#ff7f0e",
               label="Omega R0" if xi == 0 else None)
        ax.text(xi - w / 2, rec[fp]["dndx_r0"] + 0.01, f"{rec[fp]['dndx_r0']:.3f}",
                ha="center", fontsize=8)
        ax.text(xi + w / 2, rec[fp]["omega_r0"] + 0.01, f"{rec[fp]['omega_r0']:.3f}",
                ha="center", fontsize=8)
    ax.axhline(1.0, color="k", ls="--", lw=1, label="perfect recovery")
    ax.set_xticks(x)
    ax.set_xticklabels(fps)
    ax.set_ylabel("MOCK recovery ratio R0 = est/truth")
    ax.set_title("MOCK 2LPT-0 recovery [19.5,20.3)\n(justifies loa0 as headline)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, axis="y")

    ax2 = axes[1]
    if (not A["is_mock"]) and os.path.exists(REAL_LOA_ARTIFACT_PM):
        pmA = load_and_adapt(REAL_LOA_ARTIFACT_PM)
        groups = [("dN/dX", "dndx", 1.0), ("1e3*Omega", "omega", 1e3)]
        gx = np.arange(len(groups))
        for j, (lab, kind, sc) in enumerate(groups):
            v_lo = A["integ"].get(A["headline_fp"], {}).get(kind, {}).get("point")
            v_pm = pmA["integ"].get(pmA["headline_fp"], {}).get(kind, {}).get("point")
            if v_lo is not None:
                ax2.bar(gx[j] - w / 2, v_lo * sc, w, color="#1f77b4",
                        label="loa0" if j == 0 else None)
            if v_pm is not None:
                ax2.bar(gx[j] + w / 2, v_pm * sc, w, color="#2ca02c",
                        label="purity_mixture" if j == 0 else None)
        ax2.set_xticks(gx)
        ax2.set_xticklabels([g[0] for g in groups])
        ax2.set_title("REAL-LOA integrated [19.5,20.3): loa0 vs purity_mixture")
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.25, axis="y")
        # NOTE: no numeric labels on the real panel (privacy).
    else:
        ax2.axis("off")
        ax2.text(0.5, 0.5,
                 ("SMOKE/MOCK mode: real-LOA FP bracket not loaded.\nOn real data set "
                  "REAL_LOA_ARTIFACT_PM to overlay\nloa0 vs purity_mixture integrated dN/dX & Omega."
                  if A["is_mock"] else
                  "purity_mixture real-LOA artifact not found.\nSet REAL_LOA_ARTIFACT_PM to draw the "
                  "loa0-vs-pm bracket."),
                 ha="center", va="center", fontsize=9)
    _finish(fig, "fig3_fp_bracket.png")


figure_3_fp_bracket(ART)


## Figure 4 — Continuity with the DLA tier (sub-DLA -> DLA)

**Shows:** one continuous CDDF f(N) spanning the sub-DLA band into the DLA tier, with the
**20.3 tier boundary** marked. Three region types are carried across the boundary so the
reader sees what is measured vs extrapolated vs non-identifiable: `[19.5,19.7)` hatched
(non-identifiable), `[19.7,20.3)` sub-DLA measurement, `>=20.3` DLA tier. **The DLA-tier
numbers come from `UNBLIND_01` (Team 1)** — overlaid only if `DLA_ARTIFACT` is set. If the
plotted z-bin is `beyond_calibration`, the whole sub-DLA curve is styled as extrapolated.

**Falsified if:** the CDDF is discontinuous across 20.3 for a well-supported z-bin (beyond
the expected turnover), or a non-identifiable / extrapolated region is drawn as measured.


In [ ]:
def _perz_from(A2):
    # first z-bin that is BOTH not-extrapolated AND has a finite per-z CDDF (i.e. z<=3.5;
    # bins above v2_z_fit_hi have perz_fN_empty and cannot show a continuous CDDF).
    pf = A2.get("perz_fN")
    if not pf:
        return None
    centers = np.asarray(pf["logN_centers"], float)
    for z in pf.get("perz", []):
        if z.get("extrapolated"):
            continue
        f = np.asarray(z.get("f", []), float)
        if np.isfinite(f).sum() > 0:
            return centers, f, z.get("z")
    return None


def figure_4_continuity(A):
    fig, ax = plt.subplots(figsize=(7.8, 5.0))
    hi = NONIDENT_LOGN_HI
    have_dla = bool(DLA_ARTIFACT and os.path.exists(DLA_ARTIFACT))
    if A["is_mock"]:
        fp = A["headline_fp"]
        d = A["diff_marginal"][fp]
        mid, f = d["logN_mid"], d["f"]
        ok = ~nonident_mask_edges(d["logN_lo"], d["logN_hi"])
        ax.step(mid[ok], f[ok], where="mid", color="#1f77b4", lw=2, label="sub-DLA f(N) (this NB, mock)")
        ax.plot(mid[ok], f[ok], "o", color="#1f77b4", ms=5)
    else:
        got = _perz_from(A)
        if got is not None:
            centers, f, z0 = got
            ok = (~nonident_mask_centers(centers)) & (f > 0)
            ax.plot(centers[ok], f[ok], "-", color="#1f77b4", lw=2,
                    label=f"sub-DLA f(N|z~{z0:.2f}) (this NB)")
    if have_dla:
        dlaA = load_and_adapt(DLA_ARTIFACT)
        got = _perz_from(dlaA)
        if got is not None:
            centers, f, z0 = got
            ok = (centers >= 20.3) & (f > 0)
            ax.plot(centers[ok], f[ok], "-", color="#2ca02c", lw=2, label="DLA tier (UNBLIND_01)")
    # region encodings carried across the boundary
    ax.axvspan(19.5, hi, facecolor="none", edgecolor="crimson", hatch="xx", alpha=0.5, zorder=0,
               label="[19.5,19.7) non-identifiable")
    ax.axvline(20.3, color="k", ls="-.", lw=1.5)
    ax.text(20.31, 0.97, "20.3 sub-DLA | DLA boundary", transform=ax.get_xaxis_transform(),
            rotation=90, va="top", ha="left", fontsize=8)
    ax.set_yscale("log")
    ax.set_xlabel(r"$\log_{10} N_{\rm HI}$")
    ax.set_ylabel(r"$f(N_{\rm HI})$")
    ax.set_xlim(19.45, 21.6)
    ax.set_title("Continuity of the CDDF: sub-DLA -> DLA (one continuous f(N))")
    ax.legend(fontsize=8, loc="best")
    ax.grid(alpha=0.25, which="both")
    print("[fig4] DLA-tier numbers come from UNBLIND_01 (Team 1). "
          + ("DLA artifact overlaid." if have_dla else "DLA artifact not loaded; boundary marked at 20.3."))
    if not A["is_mock"]:
        print("[fig4] CAVEAT: only z<=3.5 bins have a per-z CDDF, so ONLY they can show a "
              "continuous f(N) across the 20.3 boundary. Bins above v2_z_fit_hi have no per-z "
              "CDDF (perz_fN_empty) — this panel is NOT the whole z story.")
    print("[fig4] For a z>4 bin the sub-DLA portion additionally suffers beyond_calibration AND "
          "perz_fN_empty (see the z>4 section) — not covered by the MC band.")
    _finish(fig, "fig4_continuity.png")


figure_4_continuity(ART)


## Figure 5 — Systematics summary (as data)

**Shows:** each error term, whether it is **INSIDE** the plotted band, drawn as a
**SEPARATE/OUTSIDE** band, or **EXCLUDED**, and its provenance. Any term that cannot be
substantiated from a committed routine / stamped artifact is flagged **UNVERIFIED**.
The rows are also exposed as the Python list `SYSTEMATICS_TABLE` for downstream use.

**Falsified if:** a term marked INSIDE the band is not actually reflected in the plotted
MC band, or an UNVERIFIED term is quietly promoted to a headline error.


In [ ]:
def figure_5_systematics_table(A):
    edge = _load_edge_systematic()
    fp = A["headline_fp"]
    ig = A["integ"].get(fp, {})
    has_mc = ig.get("dndx", {}).get("q16") is not None
    rows = [
        dict(term="Statistical MC band [19.5,20.3]",
             source="headline artifact integrated q16/q84,q025/q975",
             placement="INSIDE plotted band",
             status=("VERIFIED" if has_mc else "UNVERIFIED (mock: point-only, no MC band)")),
        dict(term="Non-identifiable [19.5,19.7) mass",
             source="design: 19.5-floored catalog (Track A); bins hatched, never measured",
             placement="EXCLUDED (hatched; not in band)",
             status="VERIFIED (mask honoured)"),
        dict(term="Edge-migration systematic [19.5,20.3]",
             source=("subdla_edge_systematic.json bracket (mock-derived; PANEL-ADJUDICATED)"
                     if edge is not None else "subdla_edge_systematic.json ABSENT"),
             placement="OUTSIDE (separate band)",
             status=("UNVERIFIED (mock bracket; final number panel-adjudicated)"
                     if edge is not None else "UNVERIFIED (artifact absent; TOTAL ERROR INCOMPLETE)")),
        dict(term="FP-model choice (loa0 vs purity_mixture)",
             source="subdla_mock_validation.json recovery R0 (MOCK: loa0 vs pm)",
             placement="OUTSIDE (reported as loa0|pm bracket)",
             status="VERIFIED (mock validation)"),
        dict(term="z>4 completeness extrapolation (beyond_calibration)",
             source="no mock validates z>4 (max_truth_z ~ 3.5-3.79); unbounded",
             placement="OUTSIDE (NOT covered by MC band)",
             status="UNVERIFIED (unbounded; no calibration support)"),
        dict(term="beyond v2 mean-flux fit ceiling (z>3.5)",
             source="v2_z_fit_hi=3.5; z=[3.5,4.0) bin entirely above the fit ceiling",
             placement="OUTSIDE (flagged per z-bin)",
             status="UNVERIFIED (mean-flux model extrapolation)"),
        dict(term="Per-z CDDF absent above z=3.5 (perz_fN_empty)",
             source="fine z-fit grid ends at v2_z_fit_hi=3.5; f(N|z) all-NaN above it "
                    "(dN/dX & Omega still exist)",
             placement="N/A (data-availability limit; not an error term)",
             status="STRUCTURAL (raising v2_z_fit_hi is a PI-gated science decision; not re-run here)"),
        dict(term="Deep-tail / forest-transfer systematic (~ONE-SIDED downward on Omega (12.8% at >=20.3; london-0 R0=0.8715). NOT symmetric. Only 1.9% on dN/dX. Status: ORPHANED (prose literal + artifact stamp cff73cb lacks its routine))",
             source="carried from prior project work (not stamped in this NB's artifacts)",
             placement="OUTSIDE (carried)",
             status="UNVERIFIED (no committed routine/stamped artifact in this NB)"),
    ]
    print("SYSTEMATICS SUMMARY (data):")
    for r in rows:
        print(f"  - [{r['status']}] {r['term']}: {r['placement']} | {r['source']}")
    fig, ax = plt.subplots(figsize=(13, 3.0))
    ax.axis("off")
    cell_text = [[r["term"], r["placement"], r["status"], r["source"]] for r in rows]
    tbl = ax.table(cellText=cell_text,
                   colLabels=["term", "placement in error budget", "status", "source"],
                   loc="center", cellLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.2)
    tbl.scale(1, 1.5)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_text_props(weight="bold")
        elif "UNVERIFIED" in cell_text[r - 1][2] and c == 2:
            cell.set_facecolor("#ffe0e0")
    ax.set_title("Sub-DLA systematics budget (INSIDE/OUTSIDE band; UNVERIFIED flagged)")
    _finish(fig, "fig5_systematics_table.png")
    return rows


SYSTEMATICS_TABLE = figure_5_systematics_table(ART)



## Before you commit

* **Clear ALL outputs** (Kernel -> Restart & Clear Output, or `jupyter nbconvert
  --clear-output`). The committed `.ipynb` must have zero outputs — executed figures
  encode real-LOA values.
* **Copy figures to the private notes repo** (`desi_gpy_dla_notes`), not the code repo.
* This notebook hard-codes **no real-LOA numbers**; every plotted quantity is read from
  the loaded artifact. Only labelled *mock* (2LPT-0) values are public.
